## Surge-Sense: Data Cleaning

Prepare the emergency department visit data for exploratory analysis and modeling. The cleaned visit-level data supports wait-time analysis; the daily visit data supports volume forecasting.

### 1. Setup and load source data

In [1]:
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')
data = pd.read_csv('/content/drive/MyDrive/er-data.csv')
data.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,Patient Id,Patient Admission Date,Patient First Inital,Patient Last Name,Patient Gender,Patient Age,Patient Race,Department Referral,Patient Admission Flag,Patient Satisfaction Score,Patient Waittime
0,145-39-5406,20-03-2024 08:47,H,Glasspool,M,69,White,NaN,False,10.0,39
1,316-34-3057,15-06-2024 11:29,X,Methuen,M,4,Native American/Alaska Native,NaN,True,NaN,27
2,897-46-3852,20-06-2024 09:13,P,Schubuser,F,56,African American,General Practice,True,9.0,55
3,358-31-9711,04-02-2024 22:34,U,Titcombe,F,24,Native American/Alaska Native,General Practice,True,8.0,31
4,289-26-0537,04-09-2024 17:48,Y,Gionettitti,Male,5,African American,Orthopedics,False,NaN,10


### 2. Remove unused fields and standardize columns

In [2]:
# Remove direct identifiers and the satisfaction field, which is not a modeling target.
data = data.drop(
    columns=['Patient First Inital', 'Patient Last Name', 'Patient Satisfaction Score']
)

# Normalize gender abbreviations before renaming columns.
data['Patient Gender'] = data['Patient Gender'].replace({'M': 'Male', 'F': 'Female'})
print(data['Patient Gender'].unique())

data = data.rename(columns={
    'Patient Id': 'patient_id',
    'Patient Admission Date': 'admission_date',
    'Patient Gender': 'patient_gender',
    'Patient Age': 'patient_age',
    'Patient Race': 'patient_race',
    'Department Referral': 'department_referral',
    'Patient Admission Flag': 'admission_flag',
    'Patient Waittime': 'patient_waittime',
})

<ArrowStringArray>
['Male', 'Female']
Length: 2, dtype: str


### 3. Prepare dates and referral values

In [3]:
data['admission_date'] = pd.to_datetime(data['admission_date'])
data['admission_day'] = data['admission_date'].dt.normalize()
data['admission_time'] = data['admission_date'].dt.time
data['day_of_week'] = data['admission_date'].dt.day_name()
data['department_referral'] = data['department_referral'].fillna('No referral.')
data = data.drop(columns='admission_date')

data.head()

/tmp/ipykernel_12962/2431120876.py:1: UserWarning: Parsing dates in %d-%m-%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  data['admission_date'] = pd.to_datetime(data['admission_date'])


,patient_id,patient_gender,patient_age,patient_race,department_referral,admission_flag,patient_waittime,admission_day,admission_time,day_of_week
0,145-39-5406,Male,69,White,No referral.,False,39,2024-03-20,08:47:00,Wednesday
1,316-34-3057,Male,4,Native American/Alaska Native,No referral.,True,27,2024-06-15,11:29:00,Saturday
2,897-46-3852,Female,56,African American,General Practice,True,55,2024-06-20,09:13:00,Thursday
3,358-31-9711,Female,24,Native American/Alaska Native,General Practice,True,31,2024-02-04,22:34:00,Sunday
4,289-26-0537,Male,5,African American,Orthopedics,False,10,2024-09-04,17:48:00,Wednesday


### 4. Create analysis features

In [4]:
data['age_category'] = pd.cut(
    data['patient_age'],
    bins=[0, 19, 34, 44, 54, 64, 80],
    labels=['Minor', 'Young Adult', 'Adult', 'Middle-aged', 'Older Adult', 'Senior'],
)

def time_category(hour):
    if 0 <= hour < 7 or hour >= 23:
        return 'Night'
    if hour < 15:
        return 'Day'
    return 'Evening'

data['time_category'] = data['admission_time'].apply(lambda value: time_category(value.hour))
data.head()

,patient_id,patient_gender,patient_age,patient_race,department_referral,admission_flag,patient_waittime,admission_day,admission_time,day_of_week,age_category,time_category
0,145-39-5406,Male,69,White,No referral.,False,39,2024-03-20,08:47:00,Wednesday,Senior,Day
1,316-34-3057,Male,4,Native American/Alaska Native,No referral.,True,27,2024-06-15,11:29:00,Saturday,Minor,Day
2,897-46-3852,Female,56,African American,General Practice,True,55,2024-06-20,09:13:00,Thursday,Older Adult,Day
3,358-31-9711,Female,24,Native American/Alaska Native,General Practice,True,31,2024-02-04,22:34:00,Sunday,Young Adult,Evening
4,289-26-0537,Male,5,African American,Orthopedics,False,10,2024-09-04,17:48:00,Wednesday,Minor,Evening


### 5. Save cleaned outputs

In [5]:
# Visit-level data for EDA and wait-time modeling.
data.to_csv('/content/drive/MyDrive/cleaned-data.csv', index=False)

# Daily visit counts for the volume-forecasting workflow.
daily_visits = (
    data.groupby('admission_day', as_index=False)
    .size()
    .rename(columns={'size': 'visit_count'})
)
daily_visits.to_csv('/content/drive/MyDrive/daily_visits.csv', index=False)
daily_visits.head()

,admission_day,visit_count
0,2023-04-01,19
1,2023-04-02,13
2,2023-04-03,14
3,2023-04-04,9
4,2023-04-05,19
